In [2]:
# ------------------------------------------------------
# 1. Import all required libraries
# ------------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

# ------------------------------------------------------
# 2. Load dataset (Kaggle environment)
# ------------------------------------------------------
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# ------------------------------------------------------
# 3. Data Cleaning
# ------------------------------------------------------

# Convert 'Sex' column to numeric
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})

# Fill missing Age
train['Age'].fillna(train['Age'].median(), inplace=True)
test['Age'].fillna(test['Age'].median(), inplace=True)

# Fill missing Embarked
train['Embarked'].fillna(train['Embarked'].mode()[0], inplace=True)
test['Embarked'].fillna(test['Embarked'].mode()[0], inplace=True)

# Fill missing Fare in test
test['Fare'].fillna(test['Fare'].median(), inplace=True)

# One-hot encode Embarked
train = pd.get_dummies(train, columns=['Embarked'])
test = pd.get_dummies(test, columns=['Embarked'])

# Align columns (important to avoid mismatch)
train, test = train.align(test, join='left', axis=1, fill_value=0)

# ------------------------------------------------------
# 4. Select features
# ------------------------------------------------------
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
            'Embarked_C', 'Embarked_Q', 'Embarked_S']

X = train[features]
y = train['Survived']

X_test = test[features]

# ------------------------------------------------------
# 5. Split data for validation
# ------------------------------------------------------
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ------------------------------------------------------
# 6. Create and Train Random Forest Model
# ------------------------------------------------------
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

rf_model.fit(X_train, y_train)

# ------------------------------------------------------
# 7. Validate model
# ------------------------------------------------------
y_pred = rf_model.predict(X_valid)
accuracy = accuracy_score(y_valid, y_pred)
print("✅ Random Forest Validation Accuracy:", accuracy)

# ------------------------------------------------------
# 8. Predict for Kaggle test data
# ------------------------------------------------------
final_predictions = rf_model.predict(X_test)

# ------------------------------------------------------
# 9. Save submission
# ------------------------------------------------------
output = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': final_predictions
})

output.to_csv("submission_random_forest.csv", index=False)
print("✅ submission_random_forest.csv saved successfully!")


✅ Random Forest Validation Accuracy: 0.8100558659217877
✅ submission_random_forest.csv saved successfully!
